In [ ]:
!pip install -q wurlitzer ninja
%load_ext wurlitzer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 3.5 MB/s eta 0:00:00


In [ ]:
import os, time
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.cpp_extension import load_inline
from torchvision import datasets, transforms

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
device = "cuda"

torch.manual_seed(0)

# Conv + Relu Fused Py Kernel

In [ ]:
import torch
import torch.nn.functional as F

def run_kernel(f, times, *args):
    for i in range(times):
        f(i, *args)

In [ ]:
# 🔹 Description:
# This function is supposed to perform a 3x3 convolution on each pixel of the input image
# and then apply the ReLU activation function to the result.
#
# In other words, this is the "kernel" part — where the actual multiply–accumulate
# operations of convolution happen.
#
# Parameters:
# - i : the index of the pixel (from 0 to N*H*W - 1)
# - x : the input tensor of shape [N, 1, H, W]
# - w : the convolution filter (weights) of shape [1, 1, 3, 3]
# - b : the bias term (a scalar tensor)
# - out : the output tensor to store results
# - N, H, W : dimensions of the input (batch size, height, width)
#
# Inside this function, students should:
# 1️ Convert i into (n, h, w) indices — to locate the correct pixel in the batch
# 2️ Compute the accumulated sum (acc) by multiplying the 3x3 neighborhood by the weights
# 3️ Add the bias term b
# 4️ Apply ReLU (if acc < 0, set it to 0)
# 5️ Store the result in out[n, 0, h, w]

def conv_relu_kernel_py(i, x, w, b, out, N, H, W):
    """
    Performs convolution and ReLU for a single pixel

    Args:
        i: pixel index (0 to N*H*W-1)
        x: input tensor [N, 1, H, W]
        w: weight tensor [1, 1, 3, 3]
        b: bias scalar
        out: output tensor to store results
        N, H, W: dimensions
    """
    # Convert linear index i to (n, h, w) coordinates
    n = i // (H * W)  # batch index
    remainder = i % (H * W)
    h = remainder // W  # height index
    w_idx = remainder % W  # width index

    # Initialize accumulator for convolution
    acc = 0.0

    # Perform 3x3 convolution with padding=1
    for kh in range(3):  # kernel height
        for kw in range(3):  # kernel width
            # Calculate input coordinates with padding
            ih = h + kh - 1  # -1 for padding offset
            iw = w_idx + kw - 1

            # Check boundaries (padding with zeros)
            if ih >= 0 and ih < H and iw >= 0 and iw < W:
                # Accumulate: input * weight
                acc += x[n, 0, ih, iw] * w[0, 0, kh, kw]

    # Add bias
    acc += b[0]

    # Apply ReLU activation
    acc = max(0.0, acc)

    # Store result
    out[n, 0, h, w_idx] = acc
    pass  # you should write your code here

In [ ]:
# Function: conv_relu_py
# ----------------------
# 🔹 Description:
# This is the higher-level wrapper function that coordinates the operation.
# It:
# 1️ Checks that the input tensors have the expected shapes (using assert)
# 2️ Creates an output tensor with the same size as x
# 3️ Calls the run_kernel function, which runs conv_relu_kernel_py
#     for each pixel index i (from 0 to N*H*W)
# 4️ Returns the output tensor
#
# So, this function organizes and launches the lower-level computation.
def conv_relu_py(x, w, b):
    """
    Wrapper function for conv_relu operation

    Args:
        x: input tensor [N, 1, H, W]
        w: weight tensor [1, 1, 3, 3]
        b: bias tensor [1]

    Returns:
        output tensor with same shape as x
    """
    # Verify input shapes
    assert x.dim() == 4 and x.size(1) == 1, "Input must be [N, 1, H, W]"
    assert w.shape == (1, 1, 3, 3), "Weight must be [1, 1, 3, 3]"
    assert b.shape == (1,), "Bias must be [1]"

    # Get dimensions
    N, C, H, W = x.shape

    # Create output tensor with same shape as input
    out = torch.zeros_like(x)

    # Total number of pixels to process
    total_pixels = N * H * W

    # Run kernel for each pixel
    run_kernel(conv_relu_kernel_py, total_pixels, x, w, b, out, N, H, W)

    return out
    pass  # you should write your code here

In [ ]:
device = "cpu"

H, W = 8, 8

x = torch.randn(4, 1, H, W, device=device)

conv = torch.nn.Conv2d(1, 1, kernel_size=3, padding=1, bias=True).to(device)

with torch.no_grad():
    conv.weight.copy_(torch.randn_like(conv.weight))
    conv.bias.copy_(torch.randn_like(conv.bias))

w = conv.weight.detach()
b = conv.bias.detach()

y_ref = F.relu(conv(x))

y_py = conv_relu_py(x, w, b)

print("Max diff:", (y_ref - y_py).abs().max().item())


Max diff: 4.76837158203125e-07


# Cuda Kernel

In [ ]:
cuda_begin = r'''
#include <torch/extension.h>
#include <c10/cuda/CUDAException.h>

#define CHECK_CUDA(x) TORCH_CHECK(x.device().is_cuda(), #x " must be a CUDA tensor")
#define CHECK_CONTIGUOUS(x) TORCH_CHECK(x.is_contiguous(), #x " must be contiguous")
#define CHECK_INPUT(x) CHECK_CUDA(x); CHECK_CONTIGUOUS(x)

inline unsigned int cdiv(unsigned int a, unsigned int b) { return (a + b - 1) / b;}
'''

cuda_src = cuda_begin + r'''

__global__ void conv_relu_kernel(
    const float* __restrict__ x,
    const float* __restrict__ w,
    const float* __restrict__ b,
    float* __restrict__ out,
    int N,
    int H,
    int W
) {

    // You should write your code here
    // Calculate global thread index
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    // Total number of pixels
    int total_pixels = N * H * W;

    // Check bounds
    if (idx >= total_pixels) return;

    // Convert linear index to (n, h, w) coordinates
    int n = idx / (H * W);  // batch index
    int remainder = idx % (H * W);
    int h = remainder / W;  // height index
    int w_idx = remainder % W;  // width index

    // Initialize accumulator for convolution
    float acc = 0.0f;

    // Perform 3x3 convolution with padding=1
    for (int kh = 0; kh < 3; kh++) {
        for (int kw = 0; kw < 3; kw++) {
            // Calculate input coordinates with padding
            int ih = h + kh - 1;  // -1 for padding offset
            int iw = w_idx + kw - 1;

            // Check boundaries (implicit padding with zeros)
            if (ih >= 0 && ih < H && iw >= 0 && iw < W) {
                // Calculate linear indices for memory access
                int x_idx = n * H * W + ih * W + iw;  // input index
                int w_idx_kernel = kh * 3 + kw;  // weight index

                // Accumulate: input * weight
                acc += x[x_idx] * w[w_idx_kernel];
            }
        }
    }

    // Add bias
    acc += b[0];

    // Apply ReLU activation (max(0, acc))
    acc = fmaxf(0.0f, acc);

    // Store result in output tensor
    out[idx] = acc;

}

torch::Tensor conv_relu_fused(torch::Tensor x,
                              torch::Tensor w,
                              torch::Tensor b) {
    CHECK_INPUT(x);
    CHECK_INPUT(w);
    CHECK_INPUT(b);

    TORCH_CHECK(x.dim() == 4, "x must be [N,C,H,W]");
    TORCH_CHECK(w.dim() == 4, "w must be [C_out,C_in,3,3]");
    TORCH_CHECK(b.dim() == 1, "b must be [C_out]");

    TORCH_CHECK(x.size(1) == 1, "only C_in=1 supported");
    TORCH_CHECK(w.size(0) == 1 && w.size(1) == 1 &&
                w.size(2) == 3 && w.size(3) == 3,
                "only 1x1x3x3 kernel supported");
    TORCH_CHECK(b.size(0) == 1, "only 1 output channel supported");

    auto x_c = x.contiguous();
    auto w_c = w.contiguous();
    auto b_c = b.contiguous();

    int N = x_c.size(0);
    int H = x_c.size(2);
    int W = x_c.size(3);

    auto out = torch::empty_like(x_c);

    int n_pix = N * H * W;
    int threads = 256;
    int blocks = cdiv(n_pix, threads);

    conv_relu_kernel<<<blocks, threads>>>(
        x_c.data_ptr<float>(),
        w_c.data_ptr<float>(),
        b_c.data_ptr<float>(),
        out.data_ptr<float>(),
        N, H, W
    );
    C10_CUDA_KERNEL_LAUNCH_CHECK();

    return out;
}
'''

cpp_src = r'''
torch::Tensor conv_relu_fused(torch::Tensor x,
                              torch::Tensor w,
                              torch::Tensor b);
'''

module = load_inline(
    name="conv_relu_fused_ext",
    cpp_sources=[cpp_src],
    cuda_sources=[cuda_src],
    functions=["conv_relu_fused"],
    extra_cuda_cflags=["-O3"],
    verbose=False,
)


In [ ]:
class FusedConvReLUFn(torch.autograd.Function):

    @staticmethod
    def forward(ctx, x, weight, bias):

        assert x.is_cuda and weight.is_cuda and bias.is_cuda
        assert x.dim() == 4 and x.size(1) == 1, "only C_in=1 supported"
        assert weight.shape == (1, 1, 3, 3), "only 1x1x3x3 kernel supported"
        assert bias.shape == (1,), "only 1 output channel supported"

        y = module.conv_relu_fused(x, weight, bias)
        ctx.save_for_backward(x, weight, bias, y)
        return y

    @staticmethod
    def backward(ctx, grad_output):
        x, weight, bias, y = ctx.saved_tensors

        # grad ReLU
        mask = (y > 0).to(grad_output.dtype)
        grad_z = grad_output * mask

        # dL/dx
        grad_x = torch.nn.grad.conv2d_input(
            x.shape, weight, grad_z, padding=1
        )
        # dL/dW
        grad_weight = torch.nn.grad.conv2d_weight(
            x, weight.shape, grad_z, padding=1
        )
        # dL/db
        grad_bias = grad_z.sum(dim=[0, 2, 3])

        return grad_x, grad_weight, grad_bias


In [ ]:
class FusedConvReLU(nn.Module):
    def __init__(self):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(1, 1, 3, 3))
        self.bias   = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        return FusedConvReLUFn.apply(x, self.weight, self.bias)


In [ ]:
class CNNBaseline(nn.Module):
    def __init__(self, num_convs=5):
        super().__init__()

        self.num_convs = num_convs
        self.conv = nn.Conv2d(1, 1, kernel_size=3, padding=1, bias=True)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(14*14, 64)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):

        for _ in range(self.num_convs):
            x = F.relu(self.conv(x))

        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


In [ ]:
class CNNFused(nn.Module):
    def __init__(self, num_convs=5):
        super().__init__()

        self.num_convs = num_convs
        self.conv = FusedConvReLU()
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(14*14, 64)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):

        for _ in range(self.num_convs):
            x = self.conv(x)

        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x


In [ ]:
batch_size = 128

transform = transforms.ToTensor()

train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_ds  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
test_loader  = torch.utils.data.DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)


100%|██████████| 9.91M/9.91M [00:00<00:00, 20.3MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 515kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.56MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 5.55MB/s]


In [ ]:
def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total


In [ ]:
def train_model_timed(model, train_loader, test_loader, device, epochs=3, lr=1e-3):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    epoch_total_times   = []
    epoch_forward_times = []
    epoch_backward_times= []
    epoch_other_times   = []
    test_accuracies     = []

    for epoch in range(1, epochs+1):
        model.train()
        running_loss = 0.0

        fwd_time = 0.0
        bwd_time = 0.0
        other_time = 0.0

        torch.cuda.synchronize()
        epoch_start = time.time()

        for images, labels in train_loader:
            step_start = time.time()

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            # ----- forward -----
            torch.cuda.synchronize()
            t0 = time.time()
            outputs = model(images)
            loss = criterion(outputs, labels)
            torch.cuda.synchronize()
            t1 = time.time()
            fwd_time += (t1 - t0)

            # ----- backward -----
            torch.cuda.synchronize()
            t2 = time.time()
            optimizer.zero_grad()
            loss.backward()
            torch.cuda.synchronize()
            t3 = time.time()
            bwd_time += (t3 - t2)

            # ----- optimizer step -----
            torch.cuda.synchronize()
            t4 = time.time()
            optimizer.step()
            torch.cuda.synchronize()
            t5 = time.time()
            other_time += (t5 - t4)

            running_loss += loss.item() * labels.size(0)

        torch.cuda.synchronize()
        epoch_end = time.time()
        epoch_time = epoch_end - epoch_start

        avg_loss = running_loss / len(train_loader.dataset)
        acc = evaluate(model, test_loader, device)

        epoch_total_times.append(epoch_time)
        epoch_forward_times.append(fwd_time)
        epoch_backward_times.append(bwd_time)
        epoch_other_times.append(other_time)
        test_accuracies.append(acc)

        print(
            f"Epoch {epoch}: "
            f"loss={avg_loss:.4f}, "
            f"test_acc={acc*100:.2f}%, "
            f"time_total={epoch_time:.2f}s "
            f"(fwd={fwd_time:.2f}s, bwd={bwd_time:.2f}s, other={other_time:.2f}s)"
        )

    return {
        "total_times": epoch_total_times,
        "forward_times": epoch_forward_times,
        "backward_times": epoch_backward_times,
        "other_times": epoch_other_times,
        "test_accuracies": test_accuracies,
    }


In [ ]:
num_convs = 2 # 2 and 10 should be tested

baseline_cudnn = CNNBaseline(num_convs).to(device)
baseline_cudnn_off = CNNBaseline(num_convs).to(device)
baseline_cpu = CNNBaseline(num_convs).to(device)
fused = CNNFused(num_convs).to(device)

In [ ]:
def copy_weights(dst, src):
    with torch.no_grad():

        dst.conv.weight.copy_(src.conv.weight)
        dst.conv.bias.copy_(src.conv.bias)
        dst.fc1.weight.copy_(src.fc1.weight)
        dst.fc1.bias.copy_(src.fc1.bias)
        dst.fc2.weight.copy_(src.fc2.weight)
        dst.fc2.bias.copy_(src.fc2.bias)

copy_weights(fused, baseline_cudnn)
copy_weights(baseline_cudnn_off, baseline_cudnn)
copy_weights(baseline_cpu, baseline_cudnn)

In [ ]:
print("=== Training baseline CNN (Conv2d + ReLU) - cuDNN  ===")
device = "cuda"

torch.backends.cudnn.enabled = True
torch.backends.cudnn.benchmark = True

stats_base1 = train_model_timed(baseline_cudnn, train_loader, test_loader, device, epochs=5)

=== Training baseline CNN (Conv2d + ReLU) - cuDNN  ===
Epoch 1: loss=2.3015, test_acc=11.35%, time_total=9.76s (fwd=2.21s, bwd=2.61s, other=0.51s)
Epoch 2: loss=2.3014, test_acc=11.35%, time_total=7.64s (fwd=0.65s, bwd=2.43s, other=0.41s)
Epoch 3: loss=2.3014, test_acc=11.35%, time_total=6.94s (fwd=0.58s, bwd=2.26s, other=0.39s)
Epoch 4: loss=2.3014, test_acc=11.35%, time_total=7.84s (fwd=0.63s, bwd=2.24s, other=0.41s)
Epoch 5: loss=2.3013, test_acc=11.35%, time_total=7.90s (fwd=0.60s, bwd=2.35s, other=0.38s)


In [ ]:
print("\n=== Training fused CNN (FusedConvReLU) ===")

stats_fused = train_model_timed(fused, train_loader, test_loader, device, epochs=5)



=== Training fused CNN (FusedConvReLU) ===
Epoch 1: loss=2.3016, test_acc=11.35%, time_total=8.12s (fwd=0.55s, bwd=3.21s, other=0.42s)
Epoch 2: loss=2.3014, test_acc=11.35%, time_total=7.94s (fwd=0.52s, bwd=2.91s, other=0.40s)
Epoch 3: loss=2.3014, test_acc=11.35%, time_total=7.90s (fwd=0.52s, bwd=2.86s, other=0.40s)
Epoch 4: loss=2.3014, test_acc=11.35%, time_total=7.38s (fwd=0.51s, bwd=2.82s, other=0.43s)
Epoch 5: loss=2.3013, test_acc=11.35%, time_total=8.00s (fwd=0.50s, bwd=3.11s, other=0.41s)


In [ ]:
print("=== Training baseline CNN (Conv2d + ReLU) - cuDNN off ===")

torch.backends.cudnn.enabled = False
torch.backends.cudnn.benchmark = False

stats_base2 = train_model_timed(baseline_cudnn_off, train_loader, test_loader, device, epochs=5)

=== Training baseline CNN (Conv2d + ReLU) - cuDNN off ===
Epoch 1: loss=2.3016, test_acc=11.35%, time_total=20.17s (fwd=7.62s, bwd=11.87s, other=0.30s)
Epoch 2: loss=2.3014, test_acc=11.35%, time_total=20.00s (fwd=7.64s, bwd=11.67s, other=0.30s)
Epoch 3: loss=2.3013, test_acc=11.35%, time_total=20.00s (fwd=7.75s, bwd=11.59s, other=0.30s)
Epoch 4: loss=2.3014, test_acc=11.35%, time_total=19.21s (fwd=7.36s, bwd=11.23s, other=0.29s)
Epoch 5: loss=2.3013, test_acc=11.35%, time_total=19.43s (fwd=7.34s, bwd=11.40s, other=0.29s)


In [ ]:
print("=== Training baseline CNN (Conv2d + ReLU) - CPU ===")
device = "cpu"

stats_base3 = train_model_timed(baseline_cpu, train_loader, test_loader, device, epochs=5)

=== Training baseline CNN (Conv2d + ReLU) - CPU ===
Epoch 1: loss=2.3016, test_acc=11.35%, time_total=12.10s (fwd=2.80s, bwd=7.58s, other=0.71s)
Epoch 2: loss=2.3014, test_acc=11.35%, time_total=12.02s (fwd=2.76s, bwd=7.50s, other=0.74s)
Epoch 3: loss=2.3013, test_acc=11.35%, time_total=12.15s (fwd=2.83s, bwd=7.60s, other=0.76s)
Epoch 4: loss=2.3014, test_acc=11.35%, time_total=12.00s (fwd=2.76s, bwd=7.37s, other=0.74s)
Epoch 5: loss=2.3014, test_acc=11.35%, time_total=12.10s (fwd=2.86s, bwd=7.58s, other=0.72s)


num=10

In [ ]:
num_convs = 10 # 2 and 10 should be tested

baseline_cudnn = CNNBaseline(num_convs).to(device)
baseline_cudnn_off = CNNBaseline(num_convs).to(device)
baseline_cpu = CNNBaseline(num_convs).to(device)
fused = CNNFused(num_convs).to(device)

In [ ]:
def copy_weights(dst, src):
    with torch.no_grad():

        dst.conv.weight.copy_(src.conv.weight)
        dst.conv.bias.copy_(src.conv.bias)
        dst.fc1.weight.copy_(src.fc1.weight)
        dst.fc1.bias.copy_(src.fc1.bias)
        dst.fc2.weight.copy_(src.fc2.weight)
        dst.fc2.bias.copy_(src.fc2.bias)

copy_weights(fused, baseline_cudnn)
copy_weights(baseline_cudnn_off, baseline_cudnn)
copy_weights(baseline_cpu, baseline_cudnn)

In [ ]:
print("=== Training baseline CNN (Conv2d + ReLU) - cuDNN  ===")
device = "cuda"

torch.backends.cudnn.enabled = True
torch.backends.cudnn.benchmark = True

stats_base1 = train_model_timed(baseline_cudnn, train_loader, test_loader, device, epochs=5)

=== Training baseline CNN (Conv2d + ReLU) - cuDNN  ===
Epoch 1: loss=1.1857, test_acc=84.93%, time_total=9.37s (fwd=1.89s, bwd=5.05s, other=0.39s)
Epoch 2: loss=0.4679, test_acc=87.53%, time_total=8.57s (fwd=1.84s, bwd=4.81s, other=0.40s)
Epoch 3: loss=0.3944, test_acc=89.05%, time_total=8.73s (fwd=1.95s, bwd=5.06s, other=0.41s)
Epoch 4: loss=0.3588, test_acc=89.30%, time_total=9.76s (fwd=2.04s, bwd=5.52s, other=0.38s)
Epoch 5: loss=0.3347, test_acc=90.73%, time_total=9.92s (fwd=2.08s, bwd=5.48s, other=0.41s)


In [ ]:
print("\n=== Training fused CNN (FusedConvReLU) ===")

stats_fused = train_model_timed(fused, train_loader, test_loader, device, epochs=5)


=== Training fused CNN (FusedConvReLU) ===
Epoch 1: loss=1.1679, test_acc=84.20%, time_total=10.27s (fwd=1.31s, bwd=7.19s, other=0.40s)
Epoch 2: loss=0.4696, test_acc=88.36%, time_total=11.69s (fwd=1.41s, bwd=8.34s, other=0.43s)
Epoch 3: loss=0.3990, test_acc=88.72%, time_total=10.99s (fwd=1.36s, bwd=7.92s, other=0.41s)
Epoch 4: loss=0.3590, test_acc=89.90%, time_total=11.00s (fwd=1.36s, bwd=7.86s, other=0.42s)
Epoch 5: loss=0.3264, test_acc=91.10%, time_total=10.18s (fwd=1.29s, bwd=7.12s, other=0.39s)


In [ ]:
print("=== Training baseline CNN (Conv2d + ReLU) - cuDNN off ===")

torch.backends.cudnn.enabled = False
torch.backends.cudnn.benchmark = False

stats_base2 = train_model_timed(baseline_cudnn_off, train_loader, test_loader, device, epochs=5)

=== Training baseline CNN (Conv2d + ReLU) - cuDNN off ===
Epoch 1: loss=1.2801, test_acc=84.30%, time_total=78.72s (fwd=26.26s, bwd=51.70s, other=0.36s)
Epoch 2: loss=0.4958, test_acc=86.77%, time_total=78.72s (fwd=26.39s, bwd=51.57s, other=0.36s)
Epoch 3: loss=0.4326, test_acc=87.27%, time_total=77.47s (fwd=25.78s, bwd=50.93s, other=0.35s)
Epoch 4: loss=0.3954, test_acc=88.90%, time_total=77.86s (fwd=26.00s, bwd=51.09s, other=0.35s)
Epoch 5: loss=0.3625, test_acc=89.44%, time_total=77.91s (fwd=26.01s, bwd=51.15s, other=0.36s)


In [ ]:
print("=== Training baseline CNN (Conv2d + ReLU) - CPU ===")
device = "cpu"

stats_base3 = train_model_timed(baseline_cpu, train_loader, test_loader, device, epochs=5)

=== Training baseline CNN (Conv2d + ReLU) - CPU ===
Epoch 1: loss=1.1497, test_acc=85.56%, time_total=40.69s (fwd=8.11s, bwd=31.79s, other=0.42s)
Epoch 2: loss=0.4620, test_acc=88.64%, time_total=40.23s (fwd=7.96s, bwd=31.50s, other=0.43s)
Epoch 3: loss=0.3931, test_acc=89.47%, time_total=40.51s (fwd=8.12s, bwd=31.52s, other=0.45s)
Epoch 4: loss=0.3518, test_acc=90.82%, time_total=40.00s (fwd=8.05s, bwd=31.15s, other=0.45s)
Epoch 5: loss=0.3186, test_acc=91.06%, time_total=40.03s (fwd=7.93s, bwd=31.31s, other=0.45s)
